In [0]:
import pandas as pd
from pyspark.sql import functions as F

schema = 'finance'
table_name = 'dim_taxonomy'

dbutils.widgets.text("year", "", "GAAP Version Year")
gaap_year_to_process = dbutils.widgets.get("year")
dbutils.widgets.text("target_catalog", "", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")

gaap_version = f"us-gaap/{gaap_year_to_process}"

# CHANGED: Removed linkrole filter so all link roles are included
df = spark.table("operations.finance_staging.dim_taxonomy_staging").filter(
    F.col("gaap_version") == gaap_version
).toPandas()

fact = spark.table("operations.finance_staging.fact_staging_financial_statement").select("terse_label").distinct().toPandas()
leaf_nodes = set(fact['terse_label'])

# Build parent map (unchanged — already keyed on linkrole)
parent_map = {
    (row["child_label"], row["gaap_version"], row["linkrole"]): row["parent_label"]
    for _, row in df.iterrows()
}

# CHANGED: Track which (version, linkrole) combos each child appears in
child_version_linkrole_map = (
    df.groupby("child_label")
    .apply(lambda x: list(zip(x["gaap_version"], x["linkrole"])))
    .to_dict()
)

def build_path(child, gaap_version, linkrole, parent_map, max_depth=50):
    path = []
    current = child
    visited = set()
    for _ in range(max_depth):
        key = (current, gaap_version, linkrole)
        if current is None or key in visited:
            break
        path.append(current)
        visited.add(key)
        current = parent_map.get(key)
    return path[::-1]

# CHANGED: Loop over (version, linkrole) pairs and carry linkrole into each path record
paths = []
for leaf in leaf_nodes:
    version_linkrole_pairs = child_version_linkrole_map.get(leaf, [])
    for version, linkrole in version_linkrole_pairs:
        path = build_path(leaf, version, linkrole, parent_map)
        paths.append({
            "leaf_node": leaf,
            "gaap_version": version,
            "linkrole": linkrole,   # ADDED
            "path": path
        })

paths_df = pd.DataFrame(paths)
max_depth = paths_df["path"].apply(len).max()

for i in range(max_depth):
    paths_df[f"level_{i}"] = paths_df["path"].apply(
        lambda x, i=i: x[i] if i < len(x) else None
    )

label_map = dict(zip(df["child_label"], df["child_label"]))
for col in [c for c in paths_df.columns if c.startswith("level_")]:
    paths_df[col] = paths_df[col].map(label_map)

spark.createDataFrame(paths_df).createOrReplaceTempView('df')

# CHANGED: Added linkrole and its surrogate keys to the final select
final_df = spark.sql(f"""
select
     bigint(substr(xxhash64(concat_ws('|', leaf_node)), 1, 18))        AS terse_label_bigint_key
    ,sha2(concat_ws('|', leaf_node), 256)                               AS terse_label_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', gaap_version)), 1, 18))      AS gaap_version_bigint_key
    ,sha2(concat_ws('|', gaap_version), 256)                            AS gaap_version_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', linkrole)), 1, 18))          AS linkrole_bigint_key
    ,sha2(concat_ws('|', linkrole), 256)                                AS linkrole_key_hash
    ,gaap_version
    ,linkrole
    ,leaf_node                                                           AS terse_label
    ,level_1                                                             AS terse_label_level_1
    ,level_2                                                             AS terse_label_level_2
    ,level_3                                                             AS terse_label_level_3
    ,level_4                                                             AS terse_label_level_4
    ,level_5                                                             AS terse_label_level_5
    ,level_6                                                             AS terse_label_level_6
    ,level_7                                                             AS terse_label_level_7
    ,level_8                                                             AS terse_label_level_8
    ,level_9                                                             AS terse_label_level_9
    ,level_10                                                            AS terse_label_level_10
    ,level_11                                                            AS terse_label_level_11
from df
""")

final_df.write.mode("append").option("overwriteSchema","true").saveAsTable(f"{target_catalog}.{schema}.{table_name}")